In [13]:
# PyTorch provides Metal Performance Shaders (MPS) as a backend for GPU acceleration on macOS
# !pip install torch torchvision torchaudio --pre --extra-index-url https://download.pytorch.org/whl/nightly/cpu

In [3]:
import torch

if torch.backends.mps.is_available():
    print("MPS is available!")
else:
    print("MPS is not available.")


MPS is available!


In [9]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.models.vision_transformer import vit_b_16

import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==============================
# Configurations
# ==============================
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

base_path = './AdvCelebA'
label_file = os.path.join(base_path, 'attack_CelebA.txt')
image_dir = os.path.join(base_path, 'images')
partition_file = os.path.join(base_path, 'list_eval_partition_no_overlap.txt')

# ==============================
# Load Labels and Partitions
# ==============================
labels, paths, partitions = [], [], []

with open(label_file, 'r') as f:
    for line in f:
        parts = line.strip().split()
        paths.append(os.path.join(image_dir, parts[0]))
        labels.append(int(parts[1]))

with open(partition_file, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 2:
            partitions.append(int(parts[1]))

train_paths, val_paths, test_paths = [], [], []
train_labels, val_labels, test_labels = [], [], []

for path, label, partition in zip(paths, labels, partitions):
    if partition == 0:
        train_paths.append(path)
        train_labels.append(label)
    elif partition == 1:
        val_paths.append(path)
        val_labels.append(label)
    elif partition == 2:
        test_paths.append(path)
        test_labels.append(label)

print(f"Training set size: {len(train_paths)}")
print(f"Validation set size: {len(val_paths)}")
print(f"Test set size: {len(test_paths)}")

# ==============================
# Custom Dataset
# ==============================
class CustomDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx], cv2.IMREAD_COLOR)
        img = img.astype(np.float32) / 255.0
        if self.transform:
            img = self.transform(image=img)['image']
        return img.to(torch.float32), self.labels[idx]

# ==============================
# Transformations
# ==============================
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Rotate(limit=15),
    A.Resize(224, 224),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    ToTensorV2()
])

train_dataset = CustomDataset(train_paths, train_labels, transform=train_transform)
val_dataset = CustomDataset(val_paths, val_labels, transform=val_transform)
test_dataset = CustomDataset(test_paths, test_labels, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=0)

# ==============================
# Model Setup
# ==============================
model = vit_b_16(weights="IMAGENET1K_V1")
model.heads.head = nn.Linear(model.heads.head.in_features, len(set(labels)))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("Mixed precision training disabled for MPS.")

# ==============================
# Training Loop with Early Stopping
# ==============================
def train_model(model, train_loader, val_loader, epochs=30):
    history = {'train_loss': [], 'train_accuracy': [], 'val_loss': [], 'val_accuracy': []}
    best_val_acc = 0.0
    patience_counter = 0
    patience_limit = 5

    for epoch in range(epochs):
        model.train()
        total_loss_train, correct_train = 0.0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss_train += loss.item() * images.size(0)
            correct_train += (outputs.argmax(dim=1) == labels).sum().item()

        train_loss = total_loss_train / len(train_loader.dataset)
        train_acc = correct_train / len(train_loader.dataset)

        model.eval()
        total_loss_val, correct_val = 0.0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                total_loss_val += loss.item() * images.size(0)
                correct_val += (outputs.argmax(dim=1) == labels).sum().item()

        val_loss = total_loss_val / len(val_loader.dataset)
        val_acc = correct_val / len(val_loader.dataset)

        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience_limit:
                print("Early stopping triggered.")
                break

# ==============================
# Start Training
# ==============================
train_model(model=model, train_loader=train_loader, val_loader=val_loader)


Using device: mps
Training set size: 162467
Validation set size: 20170
Test set size: 19750
Mixed precision training disabled for MPS.


Epoch 1/30: 100%|█████████████████████████| 5078/5078 [3:28:55<00:00,  2.47s/it]


Epoch 1: Train Loss=0.3584, Train Acc=0.8479, Val Loss=0.4133, Val Acc=0.8163


Epoch 2/30:   1%|▏                          | 38/5078 [00:53<1:59:21,  1.42s/it]


KeyboardInterrupt: 